# 02 - Data Profiling

Este notebook ejecuta una revision inicial de archivos locales ubicados en Landing.

No descarga datos externos, no transforma archivos y no construye Bronze, Silver ni Gold.

Si Landing esta vacio, el resultado esperado es un profiling vacio controlado.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd

from src.common.paths import LANDING_DIR, REPORTS_DIR
from src.quality.profile_sources import find_supported_files, profile_file, write_json_report

print(f'Landing dir: {LANDING_DIR}')
print(f'Reports dir: {REPORTS_DIR}')

## Buscar archivos locales soportados

El notebook busca archivos existentes en Landing. En esta fase puede no existir ningun archivo, porque la ingesta real todavia no se implementa.

In [ ]:
files = find_supported_files(LANDING_DIR)

if not files:
    print('No se encontraron archivos soportados en Landing.')
    print('El profiling real queda pendiente hasta implementar la ingesta.')
else:
    for file_path in files:
        print(file_path)

files

## Ejecutar profiling

Se perfila cada archivo encontrado usando una lectura controlada de filas.

In [ ]:
MAX_ROWS = 10000

profiles = [profile_file(file_path=file_path, max_rows=MAX_ROWS) for file_path in files]

output_path = REPORTS_DIR / 'profiling_summary.json'
write_json_report(profiles=profiles, output_path=output_path)

print(f'Perfiles generados: {len(profiles)}')
print(f'Reporte generado: {output_path}')

## Resumen por archivo

In [ ]:
profile_summary_rows = [
    {
        'file_name': profile.file_name,
        'file_extension': profile.file_extension,
        'row_count': profile.row_count,
        'column_count': profile.column_count,
        'duplicate_rows': profile.duplicate_rows,
        'error': profile.error,
    }
    for profile in profiles
]

profile_summary_df = pd.DataFrame(profile_summary_rows)
profile_summary_df

## Resumen por columna

In [ ]:
column_rows = []

for profile in profiles:
    for column in profile.columns:
        column_rows.append(
            {
                'file_name': profile.file_name,
                'column_name': column.column_name,
                'inferred_dtype': column.inferred_dtype,
                'non_null_count': column.non_null_count,
                'null_count': column.null_count,
                'null_rate': column.null_rate,
                'unique_count': column.unique_count,
                'sample_values': column.sample_values,
            }
        )

columns_profile_df = pd.DataFrame(column_rows)
columns_profile_df

## Criterio de interpretacion

- Si no existen archivos locales, el profiling real queda pendiente.
- Los tipos inferidos son preliminares.
- Una columna unica en la muestra no necesariamente es llave definitiva.
- Las reglas finales de calidad se definiran despues de observar datos reales.
- Este notebook no debe usarse para versionar datos reales.